# FishCheck — YOLOv8 어종 탐지 모델 재학습

**데이터**: Roboflow FishCheck v3 (광어 / 가자미류 + head_eye 클래스, 442장)  
**모델**: YOLOv8s Object Detection (PyTorch)  
**환경**: Google Colab T4 GPU 또는 로컬 GPU

## 실행 순서
1. 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 (로컬 GPU면 그냥 실행)
2. Section 2 에서 `API_KEY` 를 본인 Roboflow API 키로 수정
3. 셀 순서대로 전체 실행 (Ctrl+F9)
4. 학습 완료 후 Section 7 에서 best.pt 저장

## 1. 환경 설정 및 GPU 확인

In [ ]:
!pip install -q ultralytics roboflow huggingface-hub python-dotenv

import torch
from ultralytics import YOLO

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  GPU 없음 — 런타임 유형을 T4 GPU 로 변경하세요')

## 2. Roboflow 데이터셋 다운로드 (v3)

In [ ]:
import os
from pathlib import Path
from roboflow import Roboflow

IS_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if not IS_COLAB:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except Exception:
        pass

if IS_COLAB and not os.getenv('ROBOFLOW_API_KEY'):
    try:
        from google.colab import userdata
        os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
    except Exception:
        pass

API_KEY = os.getenv('ROBOFLOW_API_KEY', '')
if not API_KEY:
    raise ValueError("ROBOFLOW_API_KEY 가 없습니다. .env 파일 또는 Colab Secrets를 확인하세요.")

rf      = Roboflow(api_key=API_KEY)
project = rf.workspace('50seoks-workspace').project('fishcheck-jqum0')
version = project.version(3)
dataset = version.download('yolov8')

# 절대경로로 저장 — 노트북 실행 위치에 무관하게 동작
DATA_YAML = str(Path(dataset.location).resolve() / 'data.yaml')
print(f'\n데이터셋 경로: {dataset.location}')
print(f'data.yaml   : {DATA_YAML}')

In [ ]:
import yaml, os
from pathlib import Path

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

print('=== 데이터셋 구성 ===')
print(f'클래스 수  : {cfg["nc"]}')
print(f'클래스 이름: {cfg["names"]}')

for split in ['train', 'valid', 'test']:
    split_path = Path(dataset.location) / split / 'images'
    if split_path.exists():
        count = len(list(split_path.glob('*')))
        print(f'{split:6s}: {count}장')

## 3. YOLOv8s Detection 모델 학습

In [ ]:
import os, platform
from pathlib import Path

IS_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
# 로컬: DATA_YAML 기준으로 프로젝트 루트(notebooks/../) 계산
PROJECT  = '/content/fishcheck_runs' if IS_COLAB else str(Path(DATA_YAML).parents[2] / 'runs' / 'fishcheck')
RUN_NAME = 'yolov8s_det_v3'

print(f'PROJECT: {PROJECT}')

model = YOLO('yolov8s.pt')

model.train(
    data    = DATA_YAML,
    epochs  = 100,
    imgsz   = 640,
    batch   = 16,
    patience= 20,
    project = PROJECT,
    name    = RUN_NAME,
    device  = 0 if torch.cuda.is_available() else 'cpu',
    workers = 0 if platform.system() == 'Windows' else 2,
    hsv_h   = 0.015,
    hsv_s   = 0.7,
    hsv_v   = 0.4,
    fliplr  = 0.5,
    mosaic  = 1.0,
)

BEST_PT = Path(PROJECT) / RUN_NAME / 'weights' / 'best.pt'
print(f'\n학습 완료 — best.pt: {BEST_PT}')

## 4. 검증 결과 확인

In [ ]:
import os, matplotlib.pyplot as plt, matplotlib.image as mpimg
from pathlib import Path
from ultralytics import YOLO

IS_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

# 이전 셀 변수가 없을 경우 직접 경로 지정
if 'BEST_PT' not in dir() or not Path(str(BEST_PT)).exists():
    if IS_COLAB:
        BEST_PT   = Path('/content/fishcheck_runs/yolov8s_det_v3/weights/best.pt')
        DATA_YAML = DATA_YAML  # Section 2에서 설정됨
    else:
        # 로컬: 가장 최근 yolov8s_det_v3* 폴더 자동 탐색
        base = Path('C:/AI/runs/detect/runs/fishcheck')
        runs = sorted(base.glob('yolov8s_det_v3*'), key=lambda p: p.stat().st_mtime, reverse=True)
        if not runs:
            raise FileNotFoundError(f"학습 결과 없음: {base}")
        RUN_NAME  = runs[0].name
        PROJECT   = str(base)
        BEST_PT   = runs[0] / 'weights' / 'best.pt'
        DATA_YAML = str(Path('notebooks/FishCheck-3/data.yaml').resolve())

print(f'BEST_PT  : {BEST_PT}')
print(f'존재여부 : {Path(str(BEST_PT)).exists()}')

best_model  = YOLO(str(BEST_PT))
val_results = best_model.val(data=DATA_YAML, verbose=True)

print(f'\nmAP@50   : {val_results.box.map50:.4f} ({val_results.box.map50*100:.1f}%)')
print(f'mAP@50-95: {val_results.box.map:.4f}  ({val_results.box.map*100:.1f}%)')
print(f'Precision: {val_results.box.mp:.4f}')
print(f'Recall   : {val_results.box.mr:.4f}')

results_png = Path(str(BEST_PT)).parents[1] / 'results.png'
if results_png.exists():
    plt.figure(figsize=(14, 5))
    plt.imshow(mpimg.imread(str(results_png)))
    plt.axis('off')
    plt.title('학습 곡선')
    plt.show()

## 5. 샘플 예측 테스트

In [ ]:
import glob, random
from PIL import Image

CLASS_KO = {
    'gwangeo'         : '광어',
    'gajami'          : '가자미/도다리',
    'gwangeo_head_eye': '광어 (눈)',
    'gajami_head_eye' : '가자미 (눈)',
}

test_images = glob.glob(f'{dataset.location}/test/images/*')
if not test_images:
    test_images = glob.glob(f'{dataset.location}/valid/images/*')

samples = random.sample(test_images, min(6, len(test_images)))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flatten(), samples):
    result = best_model(img_path, verbose=False)[0]
    ann    = result.plot()
    ax.imshow(ann[..., ::-1])
    if result.boxes and len(result.boxes):
        cls_id = int(result.boxes[0].cls[0])
        cls_en = result.names[cls_id]
        conf   = float(result.boxes[0].conf[0])
        ax.set_title(f'{CLASS_KO.get(cls_en, cls_en)} {conf*100:.1f}%', fontsize=11)
    else:
        ax.set_title('미탐지', fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 6. Google Drive 백업

In [ ]:
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    drive_models = Path('/content/drive/MyDrive/FishCheck/models')
    drive_models.mkdir(parents=True, exist_ok=True)
    shutil.copy(BEST_PT, drive_models / 'best.pt')
    print(f'Drive 백업 완료: {drive_models / "best.pt"}')
else:
    import shutil
    local_out = Path('models')
    local_out.mkdir(exist_ok=True)
    shutil.copy(BEST_PT, local_out / 'best.pt')
    print(f'로컬 저장 완료: {local_out / "best.pt"}')

## 7. Hugging Face Hub 업로드

준비: huggingface.co → Settings → Access Tokens → New token (write 권한)

In [ ]:
from huggingface_hub import HfApi, login

HF_USERNAME = '50seok'

login()
REPO_ID = f'{HF_USERNAME}/fishcheck-model'
api = HfApi()
api.create_repo(repo_id=REPO_ID, exist_ok=True)
api.upload_file(
    path_or_fileobj = str(BEST_PT),
    path_in_repo    = 'best.pt',
    repo_id         = REPO_ID,
)
print(f'업로드 완료 → {REPO_ID}')

## 8. (선택) Colab에서 직접 다운로드

In [ ]:
if IS_COLAB:
    from google.colab import files
    files.download(str(BEST_PT))
    print('브라우저 저장 대화상자를 확인하세요')
else:
    print(f'로컬 실행 중 — 모델 위치: {BEST_PT.resolve()}')